In [0]:
%sql
create connection if not exists clinical_trials_dev_conn type HTTP OPTIONS (
  host = 'https://clinicaltrials.gov',
  port = 443,
  base_path = '/api/v2',
  bearer_token = 'na'
)


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

conn = w.connections.get("clinical_trials_dev_conn")

base_url = f"{conn.options['host']}{conn.options['base_path']}"


In [0]:
dbutils.widgets.text("catalog_name", "clinical_trials_dev","clinical_trials_dev")
catalog_name = dbutils.widgets.get("catalog_name")


In [0]:

spark.sql(f"USE CATALOG `{catalog_name}`")

spark.sql("USE SCHEMA bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS clinical_trials_dev_data")

In [0]:
import requests
import json
import datetime

disease = "asthma"
pageSize = 10

url = f"{base_url}/studies?query.cond={disease}&pageSize={pageSize}"


response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Request failed with status code {response.status_code}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/clinical_trials_dev_data/clinical_trials_dev_data_{current_date}_{disease}.json",
    json.dumps(data),
    overwrite=True,
)